# CP201A Lab 4: Aggregating Estimates and Margins of Error

**Fall 2026**

Last week you pulled ACS tables at three scales and saw that every estimate comes with a
margin of error. In Monday's live demo we built a neighborhood out of census tracts and
kept the margins of error in view without combining them. Today we combine them.

By the end of lab you will have, for your own neighborhood:

* counts added up across your tracts, with margins of error combined correctly
* shares (percents) with margins of error of their own
* the same numbers for your city or county, for comparison
* a median reported honestly
* a table you can paste into P/NP #5

## Learning objectives

**Everyone**
* Pull a table for your own list of tracts and confirm that every tract came back
* Handle the Census Bureau's jam values before doing any arithmetic
* Add estimates across tracts and combine their margins of error with the root sum of squares
* Calculate a share and its margin of error with the proportion formula
* Write a function once and reuse it for the city and the county
* Report a median for a neighborhood without pretending it is something it is not
* Export a table in the shape P/NP #5 asks for

**If you want more**
* Aggregate several neighborhoods at once with `groupby`
* Pull the same tracts for 2015 to 2019 and see what changed when tracts were redrawn in 2020

**Reading:** U.S. Census Bureau (2020), *Understanding and Using American Community Survey
Data: What All Data Users Need to Know*, Section 8, Calculating Measures of Error for
Derived Estimates. Every formula in this notebook comes from there.

## 0. Before we begin

Same start as Lab 3: install the `census` package, import, and load your key from the file
you saved. You do not need the `getpass` cell again.

In [ ]:
%pip install -q census

In [ ]:
from census import Census
import pandas as pd
import numpy as np
import os

In [ ]:
# Your key was saved to a file in Lab 3. This cell reads it; you do not paste it again.
try:
    with open(os.path.expanduser('~/census_key.txt')) as f:
        api_key = f.read().strip()
    print('Key loaded. It starts with:', api_key[:4] + '...')
except FileNotFoundError:
    print('No key file found. Open Lab 3, run the cell in Section 0.1 once to save your key, then run this cell again.')

c = Census(key=api_key)

### 0.1 Squares and square roots

Today's formulas square numbers and take square roots. Python's `**` operator does both:
`x**2` squares, `x**0.5` is the square root. `numpy` has `np.sqrt()` if you prefer to read
the word. Both work on a whole column at once.

In [ ]:
print(5**2, np.square(5))     # two ways to square
print(25**0.5, np.sqrt(25))   # two ways to take a square root

# The same thing on a whole column
example = pd.DataFrame({'moe': [10, 20, 30]})
example['moe_squared'] = example['moe']**2
example

### 0.2 Writing a function

`print()` and `len()` are functions Python gives you. You can write your own. A function
has a name, takes inputs (called arguments), does something, and returns a result. Once it
is defined you can call it as many times as you like.

The structure is:

```python
def function_name(argument):
    result = something done with argument
    return result
```

The indented lines belong to the function. The text in triple quotes right after the `def`
line is a **docstring**: a note Python attaches to the function, so that `help(halve)` prints
it. `#` comments work inside a function too, and are the right tool for a note on one line.
Outside a function, stick to `#`: triple quotes there make a string that Python evaluates and
throws away, which works but is not a comment.

In [ ]:
def halve(number):
    '''Return half of the number passed in.'''
    return number / 2

print(halve(10))
print(halve(7))

In [ ]:
# EXERCISE #1: write a function called add_three that takes three numbers and returns
# their sum. Then call it with 1, 2, and 3. You should get 6.

## 1. Your neighborhood

### 1.1 Set your parameters

Everything in this notebook runs from the six lines in the next cell. Change them once and
every later cell uses your neighborhood.

The defaults are West Oakland, defined as the 13 census tracts used in *Owning Our Air: The
West Oakland Community Action Plan* (Bay Area Air Quality Management District and West
Oakland Environmental Indicators Project, 2019), compared with the City of Oakland and
Alameda County.

If your own tracts are ready (from P/NP #4), put them in now. If you are still deciding,
run the lab with the defaults today and swap your tracts in before P/NP #5. Tract numbers
are six digits: tract 4014 is `401400`, tract 4014.01 would be `401401` (Lab 3, Section 2.2).

In [ ]:
NEIGHBORHOOD_NAME = 'West Oakland'
TRACT_LIST = ['401400', '401500', '401600', '401700', '401800',
              '402200', '402400', '402500', '402600', '402700',
              '410500', '981900', '982000']
STATE = '06'          # California
COUNTY = '001'        # Alameda County
PLACE = '53000'       # Oakland
ACS_YEAR = 2024       # 2020 to 2024 5-year estimates

### 1.2 Pull every tract in the county, then keep yours

In Lab 3 we typed tract numbers into the request. Today we ask for every tract in the
county (`tract:*`) and then keep the ones on our list. It costs one extra second and it
catches two problems at once: a typo in a tract number, and a tract that was split or
renumbered in 2020, neither of which produces an error message on its own.

In [ ]:
variables_of_interest = {
    'NAME': 'NAME',
    'GEO_ID': 'GEO_ID',
    'B03002_001E': 'total',
    'B03002_001M': 'total_moe',
    'B03002_003E': 'nh_white',
    'B03002_003M': 'nh_white_moe',
    'B03002_004E': 'nh_black',
    'B03002_004M': 'nh_black_moe',
    'B03002_005E': 'nh_native',
    'B03002_005M': 'nh_native_moe',
    'B03002_006E': 'nh_asian',
    'B03002_006M': 'nh_asian_moe',
    'B03002_007E': 'nh_pi',
    'B03002_007M': 'nh_pi_moe',
    'B03002_008E': 'nh_1other',
    'B03002_008M': 'nh_1other_moe',
    'B03002_009E': 'nh_multi',
    'B03002_009M': 'nh_multi_moe',
    'B03002_012E': 'hispanic',
    'B03002_012M': 'hispanic_moe',
}

# The eight race and ethnicity groups, in the order we will report them
GROUPS = ['hispanic', 'nh_white', 'nh_black', 'nh_native', 'nh_asian', 'nh_pi', 'nh_1other', 'nh_multi']

LABELS = {
    'hispanic':  'Hispanic or Latino (any race)',
    'nh_white':  'White alone, not Hispanic',
    'nh_black':  'Black or African American alone, not Hispanic',
    'nh_native': 'American Indian and Alaska Native alone, not Hispanic',
    'nh_asian':  'Asian alone, not Hispanic',
    'nh_pi':     'Native Hawaiian and Other Pacific Islander alone, not Hispanic',
    'nh_1other': 'Some other race alone, not Hispanic',
    'nh_multi':  'Two or more races, not Hispanic',
    'total':     'Total population',
}

Three names in that cell, doing three jobs:

* `variables_of_interest` renames the Census Bureau's codes (`B03002_003E`) to short names
  (`nh_white`) as soon as the data arrive. Every column you work with, and every CSV you
  save, uses the short names.
* `GROUPS` is the list of the eight short names in the order we report them. The functions
  later in the lab use it to know which columns to turn into shares and which rows to write
  in the P/NP table, so that you do not retype eight names every time.
* `LABELS` maps each short name to the full wording that goes in a table a reader sees
  (Section 7). Short names are for code; labels are for people.

In [ ]:
# Every tract in the county, one call
df_all_tracts = pd.DataFrame(
    c.acs5.get(
        list(variables_of_interest.keys()),
        {'for': 'tract:*', 'in': f'state:{STATE} county:{COUNTY}'},
        year=ACS_YEAR
    )
).rename(columns=variables_of_interest)

print(f'{len(df_all_tracts)} tracts in county {COUNTY}.')

# Keep the ones on our list
df_tracts = df_all_tracts[df_all_tracts['tract'].isin(TRACT_LIST)].copy()

# Did every tract we asked for come back?
missing = sorted(set(TRACT_LIST) - set(df_tracts['tract']))
print(f'{len(df_tracts)} of {len(TRACT_LIST)} requested tracts found.')
if missing:
    print('Not found:', missing)
    # A tract that was split in 2020 keeps its first four digits. Show any near matches.
    stems = [t[:4] for t in missing]
    near = df_all_tracts[df_all_tracts['tract'].str[:4].isin(stems)][['NAME', 'tract']]
    print('Tracts in the county that share the first four digits:')
    print(near.to_string(index=False))

df_tracts[['NAME', 'tract', 'total', 'total_moe']]

If every tract came back, move on. If anything was listed as not found, there are two
possibilities. A typo is a typo: fix it. Or the number is from the 2010 tract list and that
tract was split or renumbered in 2020 (most tract numbers are the same in both periods,
which is why a 2010 list usually comes back almost complete and the check catches only the
tracts that changed). The near matches printed above are the 2020 pieces you probably want
instead. Confirm on a map: search the tract on Census Reporter (censusreporter.org, for
example "Census Tract 4014.01, Alameda County"), which shows the tract outline and a profile,
and record the change for your Data Notes.

### 1.3 Convert to numbers and deal with jam values

Two cleaning steps before any arithmetic, both of which you met in Lab 3.

First, the API returns everything as text, so we convert the estimate and margin-of-error
columns to numbers.

Second, jam values. The Census Bureau puts special numbers where a normal value does not
apply. The two you will meet in this course:

* `-555555555` in a margin-of-error column means the estimate is **controlled**. For a few
  big numbers, like a county's total population, the Census Bureau does not estimate from the
  survey: it adjusts the survey so it adds up to the Bureau's official population estimate
  for that county. That number was set, not sampled, so there is no sampling error, and the
  honest margin of error is **zero**. You will see it on total rows (total population, total
  housing units) for counties and many cities. The category rows under a controlled total
  still have ordinary MOEs, and tract totals are never controlled, so a neighborhood total
  always carries an MOE.
* `-666666666` in an estimate column means **no estimate could be made**, usually because
  the tract has too few people or households of that kind (a port tract with no renters
  has no median rent). The honest value is **missing**, which pandas writes as `NaN`. When
  you add up a column, pandas skips missing values, which is what we want. Its margin of
  error comes back as `-222222222`, and we treat that as missing too.

The rule from Lab 3 still holds: never do arithmetic on a jam value. So we replace them
first, once, in a function we can reuse on every table today.

In [ ]:
def clean_acs(df, id_cols=('NAME', 'GEO_ID', 'state', 'county', 'tract', 'place')):
    '''Convert estimate and MOE columns to numbers and replace jam values.

    Inputs:
    - df: a DataFrame straight from the Census API, with columns already renamed
    - id_cols: columns to leave as text (identifiers)

    Output: a cleaned copy of df.
    '''
    df = df.copy()
    numeric_cols = [col for col in df.columns if col not in id_cols]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col])

    moe_cols = [col for col in numeric_cols if col.endswith('_moe')]

    # Controlled estimate: no sampling error, so the MOE is zero
    df[moe_cols] = df[moe_cols].replace(-555555555, 0)

    # No estimate available: missing, not zero
    df[numeric_cols] = df[numeric_cols].replace([-666666666, -222222222, -333333333], np.nan)

    return df

df_tracts = clean_acs(df_tracts)
df_tracts.info()

### 1.4 How large is too large? The coefficient of variation

The **relative MOE** is the margin of error divided by the estimate. The standard way to say the
same thing is the **coefficient of variation (CV)**: the standard error divided by the estimate.
ACS MOEs are published at 90 percent, so the standard error is MOE ÷ 1.645, and

CV = (MOE ÷ 1.645) ÷ estimate = relative MOE ÷ 1.645

so a relative MOE of 81 percent is always a CV of about 49 percent.

There is no single cutoff. The Census Bureau's own release standard treats CVs above 30 percent
on key estimates as a serious quality problem. Practitioners draw user lines anywhere from 12 to
40 percent (Esri: 12 percent or less reliable, 12 to 40 use with caution, over 40 unreliable;
others use 15 and 30). Choose a rule, name it in your Data Note, and do not build a claim on an
estimate that fails it. If a tract's total is zero (a port tract), the CV is a division by zero
and Python shows `inf` or `NaN`: that is the data telling you the tract has almost no one in it.

In [ ]:
# EXERCISE #2: for your tracts, print NAME, total, and total_moe, and add two columns:
#   rel_moe = total_moe / total          (the relative MOE: how big the error is next to the estimate)
#   cv      = rel_moe / 1.645            (the coefficient of variation: SE / estimate)
# Which tract has the largest CV? Is it above 40 percent (unreliable by the common Esri rule)
# or above 30 percent (the Census Bureau's own release standard)? Keep that tract in mind; it is
# where your neighborhood's uncertainty comes from. Tip: add .round(2) to show two decimals.

## 2. From tracts to a neighborhood

### 2.1 Adding estimates is easy; adding margins of error is not

The neighborhood's Hispanic or Latino population is the sum of the tract counts. Its margin
of error is **not** the sum of the tract margins of error. Errors in different tracts do not
all point the same way, so simply adding them overstates the uncertainty.

The handbook's formula for the margin of error of a sum is the **root sum of squares**
(RSS): square each margin of error, add the squares, take the square root.

$$MOE_{sum} = \sqrt{MOE_1^2 + MOE_2^2 + \cdots + MOE_n^2}$$

Watch the difference for one column.

In [ ]:
wrong = df_tracts['hispanic_moe'].sum()
right = (df_tracts['hispanic_moe']**2).sum()**0.5

print(f'Neighborhood Hispanic or Latino estimate: {df_tracts["hispanic"].sum():,.0f}')
print(f'Sum of the tract MOEs (wrong):            {wrong:,.0f}')
print(f'Root sum of squares (right):              {right:,.0f}')

Read `(df_tracts['hispanic_moe']**2).sum()**0.5` from the inside out: square the column,
sum it, take the square root. That is the whole formula in one line.

One caveat from the handbook, which you should carry into your Data Notes: the formula
assumes the pieces being added are independent, and it tends to overstate the margin of
error when many small pieces are added together. Use as few pieces as your question needs.
In practice, for Assignment 1:

* Use a published total rather than rebuilding it. The renter share is `B25003_003 / B25003_001`
  (two published numbers), not the sum of renter counts across the nine race-specific tenure
  tables.
* If you only need "not White alone, non-Hispanic," compute it as `total` minus `nh_white`
  rather than adding the other seven groups. The MOE formula is the same root sum of squares:
  the errors add even though the estimates subtract.
  ```python
  df['other'] = df['total'] - df['nh_white']
  df['other_moe'] = (df['total_moe']**2 + df['nh_white_moe']**2)**0.5
  ```
* Keep the tract list to the tracts that are really in the neighborhood. Every tract you add
  adds to the neighborhood's margin of error.
* When you collapse income bins (B19001), collapse into the fewest bands your question needs.

### 2.2 Every column at once, wrapped in a function

Now the same thing for all the estimate columns and all the MOE columns, wrapped in a
function so we can reuse it. The function takes a DataFrame of tracts and returns a
one-row DataFrame for the neighborhood.

Two things to know before running it. First, this formula is for **counts** (and for shares
built from counts, Section 4). A median, a mean, or a rate that the Census Bureau publishes
as such does not add up this way; Section 6 covers medians. Second, read the RSS line
`(df[moe_cols]**2).sum()**0.5` left to right: the dot means "then do this to the result."
Take the MOE columns, square every value, `.sum()` each column down the rows (one number per
column), then raise to the power 0.5 for the square root.

The function returns the neighborhood as **one wide row**, with every estimate and MOE as a
column. That is the working shape: the share formulas in Section 4 run column by column, and
in Section 5 we stack the neighborhood, city, and county rows into one comparison table.
It is not the shape a reader wants. Section 7 turns it into a tall table, one line per
category, for P/NP #5.

In [ ]:
def aggregate_tracts(df, name):
    '''Add up a DataFrame of tracts into one neighborhood row.

    Estimates are summed. Margins of error are combined by root sum of squares.
    Identifier columns are dropped and NAME is set to the neighborhood name.

    Inputs:
    - df: a cleaned DataFrame with one row per tract
    - name: the neighborhood name to put in the NAME column

    Output: a DataFrame with one row.
    '''
    numeric_cols = df.select_dtypes('number').columns
    moe_cols = [col for col in numeric_cols if col.endswith('_moe')]
    est_cols = [col for col in numeric_cols if col not in moe_cols]

    estimates = df[est_cols].sum()                    # plain sum
    moes = (df[moe_cols]**2).sum()**0.5               # root sum of squares

    row = pd.concat([estimates, moes])                # one long Series
    out = pd.DataFrame(row).transpose()               # make it a one-row table
    out.insert(0, 'NAME', name)
    out.insert(1, 'n_tracts', len(df))
    return out

df_nbhd = aggregate_tracts(df_tracts, NEIGHBORHOOD_NAME)
df_nbhd

Two pandas moves in there worth knowing. `pd.concat([estimates, moes])` glues two Series
end to end into one. `pd.DataFrame(row).transpose()` turns that long Series (one value per
line) into a one-row table with the values across the top, which is the shape the rest of
our tables have. The picture below is the same idea with two DataFrames: `pd.concat` stacks
them.

<img src="concatenate.png" width="360">

## 3. Combining categories (only when you mean to)

B03002 gives you eight race and ethnicity groups. Your memo tables should show all eight.
Sometimes you will also want a combined row, for a chart or for a comparison that needs
larger numbers. That is fine, as long as you say so in your Data Note and keep the
original columns.

Combining categories is the same arithmetic as combining tracts: add the estimates, root
sum of squares for the margins of error. It works on the tract table or on the
neighborhood row; the answer is the same either way. Percents always come last.

`combine_categories()` below **adds** two columns (the new category and its MOE) and leaves
every original column in place. Nothing is replaced, so "keep the eight original columns"
means: do not delete them afterward. The function works for any table whose columns follow
the `name` and `name_moe` pattern, so the same call collapses income bins in B19001 or units
in structure in B25024. Which categories to combine, if any, depends on your table and your
question; there is no standard grouping.

In [ ]:
def combine_categories(df, new_name, cols):
    '''Add a combined category column (and its MOE) to df, keeping the originals.

    Inputs:
    - df: a DataFrame with estimate columns in cols and matching <col>_moe columns
    - new_name: name of the new column, for example 'nh_other'
    - cols: list of estimate column names to combine

    Output: df with two new columns, new_name and new_name + '_moe'.
    '''
    df = df.copy()
    moe_cols = [f'{col}_moe' for col in cols]
    df[new_name] = df[cols].sum(axis='columns')
    df[f'{new_name}_moe'] = (df[moe_cols]**2).sum(axis='columns')**0.5
    return df

# Example: one row for the four smallest groups, on the neighborhood row
df_nbhd = combine_categories(df_nbhd, 'nh_other', ['nh_native', 'nh_pi', 'nh_1other', 'nh_multi'])
df_nbhd[['NAME', 'nh_native', 'nh_pi', 'nh_1other', 'nh_multi', 'nh_other', 'nh_other_moe']]

In [ ]:
# EXERCISE #3: decide whether your memo will combine any categories. If it will, build
# the combined column here and write the sentence for your Data Note as a comment, for
# example: "American Indian and Alaska Native, Native Hawaiian and Other Pacific Islander,
# some other race, and two or more races (all not Hispanic) are combined as Other."
# If it will not, write one line saying so. Either way, keep the eight original columns.

## 4. Shares and their margins of error

A share is a count divided by its universe. The neighborhood's share of residents who are
Hispanic or Latino is `hispanic / total`. But both numbers are estimates, so the share has
a margin of error of its own. The handbook's formula for a **proportion**, where the
numerator is part of the denominator:

$$MOE_P = \frac{\sqrt{MOE_X^2 - P^2 \cdot MOE_Y^2}}{Y}$$

where $X$ is the numerator estimate, $Y$ is the denominator estimate, $P = X / Y$, and
$MOE_X$ and $MOE_Y$ are their margins of error. Two notes:

* Why the minus sign: the numerator is part of the denominator, so their errors move
  together (if the survey overcounted Hispanic residents in a tract, it probably overcounted
  the total too), and the share is a little more certain than the two margins of error would
  suggest on their own. Occasionally the subtraction produces a negative number under the
  square root, which has no square root. The handbook's rule for that case is to switch the
  minus to a plus. (The plus version is the formula for a **ratio**, where the numerator is
  not part of the denominator, such as persons per household; it gives a slightly larger,
  more cautious MOE.) The function below makes that switch for you when it is needed.
* Proportion or ratio? If the top number is part of the bottom number, it is a percentage
  (proportion) and you use the minus. If the top and bottom count different things, it is a
  ratio and you use the plus. A proportion can only run from 0 to 100 percent; a ratio can go
  above, or has units like "per household."
* Why divide by Y: the MOE of the count is in people. Dividing by the total turns it into
  percentage points, the same way dividing the count by the total turns it into a percent.
  Order of operations: square, multiply, subtract, square root, then divide by Y.
* We report shares in percent, so the share and its margin of error are both multiplied
  by 100 at the end. A margin of error on a percent is in percentage points.

First one share by hand, on the neighborhood row, so you can see each piece.

In [ ]:
x = df_nbhd['hispanic']
y = df_nbhd['total']
moe_x = df_nbhd['hispanic_moe']
moe_y = df_nbhd['total_moe']

p = x / y
moe_p = (moe_x**2 - p**2 * moe_y**2)**0.5 / y

print(f'Share Hispanic or Latino: {float(p.iloc[0])*100:.1f}%  (MOE {float(moe_p.iloc[0])*100:.1f} percentage points)')

In [ ]:
def add_shares(df, groups, total='total'):
    '''Add a percent column and its MOE for each group, as a share of total.

    Inputs:
    - df: a DataFrame with estimate columns for total and each group, plus <col>_moe columns
    - groups: list of estimate column names to turn into shares
    - total: name of the universe column (default 'total')

    Output: df with new columns pct_<group> and pct_<group>_moe, in percent.
    '''
    df = df.copy()
    y = df[total]
    moe_y = df[f'{total}_moe']
    for g in groups:
        p = df[g] / y
        under_root = df[f'{g}_moe']**2 - p**2 * moe_y**2
        # Proportion formula; ratio formula (plus sign) if the value under the root is negative
        under_root = np.where(under_root < 0, df[f'{g}_moe']**2 + p**2 * moe_y**2, under_root)
        df[f'pct_{g}'] = p * 100
        df[f'pct_{g}_moe'] = under_root**0.5 / y * 100
    return df

df_nbhd = add_shares(df_nbhd, GROUPS)

pct_cols = [c for g in GROUPS for c in (f'pct_{g}', f'pct_{g}_moe')]
df_nbhd[['NAME'] + pct_cols].round(1)

In [ ]:
# EXERCISE #4: run add_shares on df_tracts (the tract table) and compare the MOE on
# pct_hispanic for your largest tract with the MOE on pct_hispanic for the whole
# neighborhood. Which is smaller, and why? Two sentences as a comment.

## 5. The comparison geography

Now the same table for your city and your county, run through the same functions. This is
the payoff of writing functions: three lines each.

One thing to notice: the county's total population is controlled, so `clean_acs` set its
`total_moe` to zero. Look at what that does to the proportion formula above: the second
term under the root disappears, and the margin of error of the share is simply the
numerator's margin of error divided by the total. Note which number is zero: the total's
**MOE**, not the total. You still divide by the county's population, which is in the
millions. Simpler, and correct.

`pull_geo()` below is the pull from Section 1.2 written as a function. Use it whenever you
want a table from the API for one geography (or for every tract in a county), instead of
retyping the `pd.DataFrame(c.acs5.get(...))` block: give it the geography, a variables
dictionary, and a year, and it returns the table renamed and cleaned.

In [ ]:
def pull_geo(geo_for, geo_in, variables=variables_of_interest, year=ACS_YEAR):
    '''Pull one geography for one year and clean it.

    Inputs:
    - geo_for, geo_in: the geography, as in Lab 3 ('place:53000', 'state:06')
    - variables: a dictionary of variable codes to labels (default: the B03002 dictionary)
    - year: the final year of the 5-year period (default: ACS_YEAR)
    '''
    df = pd.DataFrame(
        c.acs5.get(list(variables.keys()), {'for': geo_for, 'in': geo_in}, year=year)
    ).rename(columns=variables)
    return clean_acs(df)

df_city = add_shares(pull_geo(f'place:{PLACE}', f'state:{STATE}'), GROUPS)
df_county = add_shares(pull_geo(f'county:{COUNTY}', f'state:{STATE}'), GROUPS)

# Stack the three rows into one comparison table
df_compare = pd.concat([df_nbhd, df_city, df_county], ignore_index=True)
df_compare[['NAME', 'total', 'total_moe'] + pct_cols].round(1)

Read across a row: the neighborhood's margins of error are several times the city's, and
the county's are smaller still. That is the pattern you noticed in Lab 3, now with a
number attached to it. It is also the reason Lab 5 exists: whether the neighborhood
"differs" from the city depends on whether the gap is larger than what those margins of
error allow, and that is a test, not a glance.

## 6. Medians

Median household income (B19013), median gross rent (B25064), and median house value
(B25077) are the tables everyone reaches for. They do not aggregate. A neighborhood's
median income is the middle of a distribution we do not have; it is not the sum of the
tract medians, and it is not their average either.

How the Census Bureau makes a median: for each geography it sorts the households into the
income bins of B19001, finds the bin where the running total crosses 50 percent, and
interpolates within that bin to a dollar figure. Its margin of error comes from the
uncertainty in where that 50 percent point falls. Because the median depends on the whole
distribution and not on a count, two medians cannot be added, and averaging them is only a
rough stand-in.

What to do instead, in order of preference:

1. Report the published median for the city or county (it exists), and report the
   **range** of your tract medians (lowest to highest) alongside it, for example "median
   household income in Oakland is $X; across the neighborhood's tracts it runs from $Y to
   $Z." The tract range can sit entirely above or below the city median; that is a finding
   about the neighborhood, not a problem with the numbers. This is what most agency reports
   do.
2. If you need one number for the neighborhood, report the simple average of the tract
   medians (add them, divide by the number of tracts) and the simple average of their
   margins of error, and label it in the note as an **approximation**. The label is doing
   the real work.
3. (Advanced path) Rebuild the median from the distribution table (B19001 for income,
   B25063 for rent), which is made of counts and does aggregate. We will point you to this
   in Lab 5.

A median needs enough households of that kind in the tract, so some tracts come back missing,
and the number varies by table. Median rent needs enough renter households paying cash rent,
so more tracts come back missing for rent than for income. For the West Oakland default,
expect 12 of 13 tracts with a median household income and 10 of 13 with a median gross rent.

The cell below does options 1 and 2 for median household income. Watch for `NaN`: a tract
with too few households returns `-666666666`, which `clean_acs` turned into missing, and
`.mean()` skips it. Say in your note how many tracts had a median.

In [ ]:
median_vars = {
    'NAME': 'NAME',
    'GEO_ID': 'GEO_ID',
    'B19013_001E': 'med_hh_income',
    'B19013_001M': 'med_hh_income_moe',
}

# pull_geo from Section 5 works for any dictionary and any geography, tracts included
med_all = pull_geo('tract:*', f'state:{STATE} county:{COUNTY}', median_vars)
med_tracts = med_all[med_all['tract'].isin(TRACT_LIST)].copy()
med_city = pull_geo(f'place:{PLACE}', f'state:{STATE}', median_vars)
med_county = pull_geo(f'county:{COUNTY}', f'state:{STATE}', median_vars)

print('Tract medians:')
print(med_tracts[['NAME', 'med_hh_income', 'med_hh_income_moe']].to_string(index=False))
print()
n_with_median = med_tracts['med_hh_income'].notna().sum()
print(f'{n_with_median} of {len(med_tracts)} tracts have a published median.')
print(f'Range of tract medians: ${med_tracts["med_hh_income"].min():,.0f} to ${med_tracts["med_hh_income"].max():,.0f}')
print(f'Average of tract medians (approximation): ${med_tracts["med_hh_income"].mean():,.0f} '
      f'(average MOE ${med_tracts["med_hh_income_moe"].mean():,.0f})')
print()
print(f'{med_city["NAME"].iloc[0]}: ${med_city["med_hh_income"].iloc[0]:,.0f} (MOE ${med_city["med_hh_income_moe"].iloc[0]:,.0f})')
print(f'{med_county["NAME"].iloc[0]}: ${med_county["med_hh_income"].iloc[0]:,.0f} (MOE ${med_county["med_hh_income_moe"].iloc[0]:,.0f})')

In [ ]:
# EXERCISE #5: repeat Section 6 for median gross rent (B25064) or, if your neighborhood
# is mostly owners, median house value (B25077). Four steps:
#   1. Make a new dictionary called rent_vars with the same four keys as median_vars, but with
#      B25064_001E and B25064_001M (check both codes: a copied dictionary with the old M code
#      left in is the most common mistake here).
#   2. Pull with pull_geo, using NEW names (rent_all, rent_tracts, rent_city, rent_county) so
#      you do not overwrite the income results.
#   3. Print the tract values, the range, the labeled average, and the city and county values,
#      as Section 6 does.
#   4. Write the Data Note sentence you would put under the table, as a comment.
# You do not need GROUPS or LABELS for a single median.

## 7. Export in the shape P/NP #5 asks for

P/NP #5 wants, for each geography, a table with one row per category and columns for the
count, its margin of error, the percent, and its margin of error. The function below
reshapes one geography row into that long table. We save one for the neighborhood and one
for the comparison geography, plus the wide comparison table for later.

Round when you paste into the template, not here: percents to one decimal place, counts
to whole numbers, and the same number of decimals down every column.

In [ ]:
def long_table(row, groups, labels, total='total'):
    '''Turn one geography row into a long table: one line per category.'''
    r = row.iloc[0]
    lines = []
    for g in groups:
        lines.append({
            'Category': labels[g],
            'Count': r[g],
            'Count MOE': r[f'{g}_moe'],
            'Percent': r[f'pct_{g}'],
            'Percent MOE': r[f'pct_{g}_moe'],
        })
    lines.append({'Category': labels[total] + ' (universe)', 'Count': r[total], 'Count MOE': r[f'{total}_moe'],
                  'Percent': 100.0, 'Percent MOE': np.nan})
    return pd.DataFrame(lines)

table_nbhd = long_table(df_nbhd, GROUPS, LABELS)
table_city = long_table(df_city, GROUPS, LABELS)
table_county = long_table(df_county, GROUPS, LABELS)

table_nbhd.round(1)

In [ ]:
safe_name = NEIGHBORHOOD_NAME.lower().replace(' ', '_')

table_nbhd.to_csv(f'lab4_{safe_name}_race_ethnicity.csv', index=False)
table_city.to_csv('lab4_city_race_ethnicity.csv', index=False)
table_county.to_csv('lab4_county_race_ethnicity.csv', index=False)
df_compare.to_csv(f'lab4_{safe_name}_comparison_wide.csv', index=False)
med_tracts.to_csv(f'lab4_{safe_name}_tract_medians.csv', index=False)

print('Saved. Check the file browser on the left, then download each file (right-click > Download).')
print('Lab 5 starts from the _comparison_wide.csv files.')

## 8. If you want more

### 8.1 Several neighborhoods at once with `groupby`

Optional. If your neighborhood is large and really two communities (thirteen tracts that
split along a freeway, say) and you want to compare the halves, or if you are comparing two
neighborhoods, `groupby` does the
aggregation in a few lines. The trick is to square the MOE columns first, let `groupby`
sum everything, and then take the square root of the MOE columns only. (Taking the square
root of the estimate columns too is a common mistake; the estimates are plain sums.)

In [ ]:
# Example: split the default West Oakland tracts into two parts. Replace with your own map.
parts = {
    '401400': 'North', '401500': 'North', '401600': 'North', '401700': 'North', '401800': 'North',
    '402200': 'South', '402400': 'South', '402500': 'South', '402600': 'South', '402700': 'South',
    '410500': 'South', '981900': 'South', '982000': 'South',
}

df_parts = df_tracts.copy()
df_parts['part'] = df_parts['tract'].map(parts)

numeric_cols = df_parts.select_dtypes('number').columns
moe_cols = [col for col in numeric_cols if col.endswith('_moe')]

df_parts[moe_cols] = df_parts[moe_cols]**2                 # 1. square the MOEs
df_grouped = df_parts.groupby('part')[numeric_cols].sum()  # 2. sum everything within each part
df_grouped[moe_cols] = df_grouped[moe_cols]**0.5           # 3. square root the MOEs only

df_grouped = add_shares(df_grouped, GROUPS)
df_grouped[['total', 'total_moe', 'pct_hispanic', 'pct_hispanic_moe', 'pct_nh_black', 'pct_nh_black_moe']].round(1)

### 8.2 The same tracts in 2015 to 2019

Part II of Assignment 1 compares 2015 to 2019 with 2020 to 2024. The 2019 estimates sit on
2010 tract boundaries, and some tracts were split, merged, or renumbered in 2020. The cell
below pulls your list for `year=2019`, reports any tract that does not come back, and shows
the neighborhood in both periods side by side (`n_tracts` tells you whether both rows rest
on the same number of tracts).

If every tract is found, the two rows are comparable. If some are not, the 2019 row is
incomplete and should not be used as is. Give the cell a separate 2010 list: set
`TRACT_LIST_2019` to the 2010 numbers (the starter tract file and the Tract Codebook notebook
on bCourses tell you what they are) and rerun. A tract that was split keeps working, because
the 2010 tract in the 2019 pull and its pieces in the 2024 pull cover the same land. A tract
whose boundary was adjusted needs a note, or the crosswalk companion that comes with Lab 5.

In [ ]:
# The 2010 tract numbers for the 2019 pull. They are the same as TRACT_LIST unless a tract
# was split or renumbered in 2020; if the check below reports missing tracts, list the 2010
# numbers here instead.
TRACT_LIST_2019 = TRACT_LIST

df_all_2019 = pd.DataFrame(
    c.acs5.get(
        list(variables_of_interest.keys()),
        {'for': 'tract:*', 'in': f'state:{STATE} county:{COUNTY}'},
        year=2019
    )
).rename(columns=variables_of_interest)

df_tracts_2019 = clean_acs(df_all_2019[df_all_2019['tract'].isin(TRACT_LIST_2019)].copy())
missing_2019 = sorted(set(TRACT_LIST_2019) - set(df_tracts_2019['tract']))
print(f'{len(df_tracts_2019)} of {len(TRACT_LIST_2019)} tracts found in the 2015 to 2019 estimates.')
if missing_2019:
    print('Not found in 2019:', missing_2019)

df_nbhd_2019 = add_shares(aggregate_tracts(df_tracts_2019, f'{NEIGHBORHOOD_NAME}, 2015 to 2019'), GROUPS)
df_nbhd_2024 = df_nbhd.copy()
df_nbhd_2024['NAME'] = f'{NEIGHBORHOOD_NAME}, 2020 to 2024'

pd.concat([df_nbhd_2019, df_nbhd_2024], ignore_index=True)[['NAME', 'n_tracts', 'total', 'total_moe'] + pct_cols].round(1)

## 9. Resources: finding tracts, tables, and variable codes

Bookmark these. Each one answers a different question.

**Which tracts make up my neighborhood?**
* **Census Reporter** (https://censusreporter.org/): the easiest map. Search a tract by
  name ("Census Tract 4014, Alameda County") or search an address and click through to its
  tract. Each tract page shows the outline on a map and a profile with the main tables filled
  in, which is a fast way to see whether a tract is mostly residents, a port, or a campus.
* **Census Geocoder** (https://geocoding.geo.census.gov/geocoder/): paste an address and get
  its 2020 tract number back. Useful for turning a boundary described in street names into
  tracts.
* **MTC 2020 census tracts map** (https://opendata.mtc.ca.gov/datasets/4901642d06e74e8f84d060948226b748):
  a browsable map of every 2020 tract in the nine Bay Area counties, from the Metropolitan
  Transportation Commission. Bay Area only.
* **2020 to 2010 tract relationship file** (https://www.census.gov/geographies/reference-files/time-series/geo/relationship-files.2020.html):
  the official record of which 2010 tracts became which 2020 tracts. The starter tract file
  was built from the California file, and the Tract Codebook notebook on bCourses runs the
  same check for any list you give it.

**Which table has the variable I want, and what is its code?**
* **ACS 5-year table list for 2024** (https://api.census.gov/data/2024/acs/acs5/groups.html):
  every table (group) in the vintage this course uses, with its name and universe. Search the
  page (Ctrl+F or Cmd+F) for a word like "tenure" or "poverty." Click a table to see its
  variables and their codes.
* **All Census API datasets** (https://api.census.gov/data.html): the master list. Search for
  `2024/acs/acs5` to find the 5-year dataset; the 1-year and older vintages are listed the same
  way. You will need this if a table changed between 2019 and 2024.
* **Census Reporter**, again: its table search is friendlier than the API pages, and each table
  page shows the variable layout. Confirm the code on the API groups page before you pull it.
* **Social Explorer** (https://www.socialexplorer.com/): licensed for Berkeley students through
  the Library. Good for browsing tables and making quick maps; not a source for your pulls,
  since the numbers in your memo should come from the API with their margins of error.

**What is the code for my county or city?**
* **FIPS code list** (https://transition.fcc.gov/oet/info/maps/census/fips/fips.txt): state and
  county codes, searchable in the browser. California is `06`.
* **Place codes**: the fastest way is a `place:*` pull for California (Lab 3, Section 7.1) and a
  search of the `NAME` column for your city.

**How do I read the numbers?**
* U.S. Census Bureau (2020), *Understanding and Using American Community Survey Data: What
  All Data Users Need to Know* (https://www.census.gov/programs-surveys/acs/library/handbooks/general.html):
  Section 7 for margins of error and significance, Section 8 for the formulas in this notebook.
  This is the same handbook as this week's reading on bCourses; the link is the Census Bureau's
  page for it, which is where to look for a newer edition in future years.

**Context for your neighborhood (not for estimation or testing)**
* **California Hard-to-Count Index map** (https://cacensus.maps.arcgis.com/apps/webappviewer/index.html?id=48be59de0ba94a3dacff1c9116df8b37),
  California Department of Finance: a score for every tract predicting how hard it is to count
  in the census, with the variables behind the score (young children, recent movers,
  multi-unit buildings, no broadband, and others). Click a land tract, not the water. Tracts
  that score high are the same tracts where the ACS has fewer completed interviews and wider
  margins of error, so this map often explains the size of the MOEs you saw in Section 2.
  Tract scores use 2017 to 2021 ACS data; use them as background in your introduction, not
  as numbers in your exhibits.
* **Response Outreach Area Mapper (ROAM)** (https://www.census.gov/library/visualizations/2017/geo/roam.html):
  the Census Bureau's nationwide version of the same idea, mapping its Low Response Score
  by tract, with a 2030 edition built on the 2024 Planning Database. Use it for a
  neighborhood outside California.
* **CalEnviroScreen 5.0** (https://oehha.ca.gov/calenviroscreen/report/calenviroscreen-50):
  environmental and health burden by tract. Check which tract boundaries it uses before
  matching to yours. Same rule: optional context, cited, and kept out of the estimates and
  tests.

## 10. Using this notebook for the rest of Assignment 1

You have run the whole chain once, on one table. The rest of Assignment 1 is the same chain
on other tables and, for Part II, other years. This section says exactly what to repeat and
where.

### 10.1 The recipe for any count table

0. Make this notebook yours: File > Save Notebook As, with a name like
   `assignment1_chinatown.ipynb`, and set Section 1.1 to your tracts, county, and city. Run
   every cell once so the functions are defined. Then add your own tables in Section 11 at
   the end, one dictionary and one chain per table.
1. Find the table and its variable codes (Section 9, or the starter tables in the Assignment 1
   handout). Open it and note the universe: the `_001E` variable is the total that shares are
   divided by. Pull the universe row, the rows your question needs, and the `M` for every `E`.
   If your exhibit is a distribution (race and ethnicity, income bands, age groups), pull every
   category so the parts add to the whole.
2. Write a dictionary in a new cell with a new name, for example `commute_vars`: codes on the
   left, short names on the right, keeping `NAME` and `GEO_ID`. Keep the pattern: every
   estimate name has a matching `_moe` name. Put a markdown cell above it with the table number,
   title, universe, and years; that becomes your Data Note.
3. Pull the tracts with `pull_geo('tract:*', ...)` and your dictionary, filter to `TRACT_LIST`,
   and check that every tract came back, as the tenure example below does.
4. Run the chain: `clean_acs`, then `aggregate_tracts`, then `add_shares` with `total=` set
   to your universe column and `groups` set to the columns you want as shares.
5. Pull the city and county with `pull_geo(..., variables=your_dict)` and run `add_shares`
   on each.
6. Make the P/NP-shaped table with `long_table`, using a `labels` dictionary of your own,
   and save the CSVs (Section 7).

The cell below has both chains you will need for Question 2, tenure and poverty, written out
in full. Run the one for the option you chose. Together they are also the model for any other
count table in Assignment 1: the same six steps with a different dictionary.

In [ ]:
# Question 2, Option B: tenure (B25003). Universe: occupied housing units.
tenure_vars = {
    'NAME': 'NAME',
    'GEO_ID': 'GEO_ID',
    'B25003_001E': 'units',
    'B25003_001M': 'units_moe',
    'B25003_002E': 'owner',
    'B25003_002M': 'owner_moe',
    'B25003_003E': 'renter',
    'B25003_003M': 'renter_moe',
}

# Question 2, Option A: poverty (B17001). Universe: population for whom poverty status is determined.
poverty_vars = {
    'NAME': 'NAME',
    'GEO_ID': 'GEO_ID',
    'B17001_001E': 'pov_universe',
    'B17001_001M': 'pov_universe_moe',
    'B17001_002E': 'below_poverty',
    'B17001_002M': 'below_poverty_moe',
}

# The chain, for tenure.
ten_all = pull_geo('tract:*', f'state:{STATE} county:{COUNTY}', tenure_vars)
ten_tracts = ten_all[ten_all['tract'].isin(TRACT_LIST)].copy()
print(f'{len(ten_tracts)} of {len(TRACT_LIST)} tracts found.')

ten_nbhd = add_shares(aggregate_tracts(ten_tracts, NEIGHBORHOOD_NAME), ['owner', 'renter'], total='units')
ten_city = add_shares(pull_geo(f'place:{PLACE}', f'state:{STATE}', tenure_vars), ['owner', 'renter'], total='units')
ten_county = add_shares(pull_geo(f'county:{COUNTY}', f'state:{STATE}', tenure_vars), ['owner', 'renter'], total='units')

ten_compare = pd.concat([ten_nbhd, ten_city, ten_county], ignore_index=True)
ten_compare.to_csv(f'lab4_{safe_name}_tenure_comparison_wide.csv', index=False)
ten_compare[['NAME', 'units', 'units_moe', 'renter', 'renter_moe', 'pct_renter', 'pct_renter_moe']].round(1)

# The same chain for poverty. Everything that changed from the tenure block is the dictionary,
# the total (the universe column), the groups, and the variable names.
pov_all = pull_geo('tract:*', f'state:{STATE} county:{COUNTY}', poverty_vars)
pov_tracts = pov_all[pov_all['tract'].isin(TRACT_LIST)].copy()
print(f'{len(pov_tracts)} of {len(TRACT_LIST)} tracts found.')

pov_nbhd = add_shares(aggregate_tracts(pov_tracts, NEIGHBORHOOD_NAME), ['below_poverty'], total='pov_universe')
pov_city = add_shares(pull_geo(f'place:{PLACE}', f'state:{STATE}', poverty_vars), ['below_poverty'], total='pov_universe')
pov_county = add_shares(pull_geo(f'county:{COUNTY}', f'state:{STATE}', poverty_vars), ['below_poverty'], total='pov_universe')

pov_compare = pd.concat([pov_nbhd, pov_city, pov_county], ignore_index=True)
pov_compare.to_csv(f'lab4_{safe_name}_poverty_comparison_wide.csv', index=False)
pov_compare[['NAME', 'pov_universe', 'pov_universe_moe', 'below_poverty', 'below_poverty_moe', 'pct_below_poverty', 'pct_below_poverty_moe']].round(1)

### 10.2 What you need for P/NP #5 (due Sunday, October 4)

* **Part I** (case study): your parameters from Section 1.1, the missing-tract check from
  Section 1.2, and the starter tract file or the boundary you cite.
* **Part II, Question 1** (race and ethnicity): the neighborhood and city or county tables
  from Section 7, with the Data Note ingredients from Exercise #3 and Section 1.2.
* **Part II, Question 2**: Option A needs median household income (Section 6) and the poverty
  rate (the `poverty_vars` chain above). Option B needs tenure (the `tenure_vars` chain above)
  and median gross rent or house value (Exercise #5).
* **Part III** (significance tests): next week. Lab 5 starts from the `_comparison_wide.csv`
  files you saved, converts margins of error to standard errors, and runs the test on the
  shares. Nothing more to do on it today.

### 10.3 What you need for Assignment 1 (due Sunday, October 11)

**Part I** is the three conditions questions for 2020 to 2024, neighborhood versus city or
county, so it is P/NP #5 plus the question you did not choose. Same recipe.

**Part II** is one topic track, two questions, comparing 2015 to 2019 with 2020 to 2024.
Which cells you repeat depends on your learning path:

* **New learner path (city or county scale).** No tracts and no boundary changes. Pull the
  same dictionary twice with `pull_geo(..., year=2019)` and `pull_geo(..., year=2024)`, run
  `add_shares` on each, and stack the two rows with `pd.concat`. That is the whole exhibit;
  the significance test is Lab 5.
* **Advanced path (neighborhood scale).** Section 8.2 is your template: pull the tract table
  for `year=2019`, read the missing-tract check, and use the starter tract file or the Tract
  Codebook notebook to know your 2010 numbers. If every tract is found and unchanged, run
  `aggregate_tracts` on both years. If a tract was split or renumbered, pull the 2010 numbers
  for 2019 and the 2020 numbers for 2024; the pieces add back up. If a boundary was adjusted,
  Lab 5 has a crosswalk companion notebook.
* **A mix** is one exhibit each way.

Whatever the path, two things carry over from the assignment: only compare 2015 to 2019 with
2020 to 2024 (never two periods that share a year), and treat dollar values as nominal unless
you adjust the 2019 values with the CPI-U factor given in Lab 5. Say which in your Data Note.

### 10.4 Save your work outside Datahub

Your CSVs and this notebook live in your Datahub account, which is not a backup. Before you
leave, download the CSVs you saved (right-click a file in the browser on the left, then
Download) and the notebook itself (File > Download). You will need the CSVs for Lab 5, and you
will want the notebook when you write the memo.

## 11. Your tables

Add your own tables here, one per block: a markdown cell with the table number, title,
universe, and years, then a dictionary, then the chain. The example below is commute mode,
B08301 (Means of Transportation to Work; universe: workers 16 years and over; 2020 to 2024
ACS 5-Year Estimates). It pulls only the rows a commute question needs. Confirm every code on
the 2024 groups page before you pull, and for Part II check the 2019 page too: the transit
sub-mode rows of this table were relabeled between the two vintages.

In [ ]:
commute_vars = {
    'NAME': 'NAME',
    'GEO_ID': 'GEO_ID',
    'B08301_001E': 'workers',       # universe: workers 16 and over
    'B08301_001M': 'workers_moe',
    'B08301_003E': 'drove_alone',
    'B08301_003M': 'drove_alone_moe',
    'B08301_010E': 'transit',
    'B08301_010M': 'transit_moe',
    'B08301_018E': 'bike',
    'B08301_018M': 'bike_moe',
    'B08301_019E': 'walked',
    'B08301_019M': 'walked_moe',
    'B08301_021E': 'wfh',
    'B08301_021M': 'wfh_moe',
}
COMMUTE_GROUPS = ['drove_alone', 'transit', 'bike', 'walked', 'wfh']

com_all = pull_geo('tract:*', f'state:{STATE} county:{COUNTY}', commute_vars)
com_tracts = com_all[com_all['tract'].isin(TRACT_LIST)].copy()
print(f'{len(com_tracts)} of {len(TRACT_LIST)} tracts found.')

com_nbhd = add_shares(aggregate_tracts(com_tracts, NEIGHBORHOOD_NAME), COMMUTE_GROUPS, total='workers')
com_city = add_shares(pull_geo(f'place:{PLACE}', f'state:{STATE}', commute_vars), COMMUTE_GROUPS, total='workers')
com_county = add_shares(pull_geo(f'county:{COUNTY}', f'state:{STATE}', commute_vars), COMMUTE_GROUPS, total='workers')

com_compare = pd.concat([com_nbhd, com_city, com_county], ignore_index=True)
com_compare.to_csv(f'lab4_{safe_name}_commute_comparison_wide.csv', index=False)
com_compare[['NAME', 'workers'] + [c for g in COMMUTE_GROUPS for c in (f'pct_{g}', f'pct_{g}_moe')]].round(1)

## Before you leave

* You can pull a table for your own tracts, check that they all came back, and clean it.
* You can add up tracts with the right margin of error, and calculate shares with theirs.
* You have one function for each step, and you have used them on three geographies.
* Your CSVs and notebook are downloaded, not just saved in Datahub.

**Coming up:** Monday and Wednesday next week cover statistical significance: when two
estimates with margins of error are actually different. Lab 5 (September 30 and October 2)
runs those tests on the numbers you made today, then gives you open time on Assignment 1.
**P/NP #5 is due Sunday, October 4.**